In [ ]:
from google.colab import files
uploaded = files.upload()   # returns a dict, not a DataFrame

import pandas as pd

# Now read the uploaded file into a DataFrame
loan = pd.read_csv("Loan.txt", sep=None, engine="python")

print(loan.head())
print(loan.shape)
print("Loaded Loan.txt and previewed head/shape")


Saving Loan.txt to Loan (1).txt
    loanId  memberId        date            purpose  isJointApplication  \
0  1888978   2305095  12/10/2014  debtconsolidation                 0.0   
1  1299695   2610493   9/15/2014  debtconsolidation                 0.0   
2  1875016   2491679   9/11/2014  debtconsolidation                 0.0   
3  1440478   2092798   4/22/2016    homeimprovement                 0.0   
4  1124634   2633077    2/3/2016  debtconsolidation                 0.0   

   loanAmount       term  interestRate  monthlyPayment grade loanStatus  
0     25190.0  60 months          6.25             490    E3    Current  
1     21189.0  60 months         10.49             455    B3    Current  
2     29908.0  60 months          9.11             622    B2    Current  
3     13053.0  48 months         11.89             343    B3    Current  
4     24613.0  60 months         15.13             587    A3    Current  
(100000, 11)
Loaded Loan.txt and previewed head/shape


In [ ]:

# Trim whitespace, standardize casing for key text columns
loan_clean = loan.copy()

for col in loan_clean.select_dtypes(include=['object']).columns:
    loan_clean[col] = loan_clean[col].astype(str).str.strip()

if 'purpose' in loan_clean.columns:
    loan_clean['purpose'] = loan_clean['purpose'].str.lower()

if 'grade' in loan_clean.columns:
    loan_clean['grade'] = loan_clean['grade'].str.upper()

print(loan_clean.head())
print('Cleansed strings: trimmed and standardized text case')

    loanId  memberId        date            purpose  isJointApplication  \
0  1888978   2305095  12/10/2014  debtconsolidation                 0.0   
1  1299695   2610493   9/15/2014  debtconsolidation                 0.0   
2  1875016   2491679   9/11/2014  debtconsolidation                 0.0   
3  1440478   2092798   4/22/2016    homeimprovement                 0.0   
4  1124634   2633077    2/3/2016  debtconsolidation                 0.0   

   loanAmount       term  interestRate  monthlyPayment grade loanStatus  
0     25190.0  60 months          6.25             490    E3    Current  
1     21189.0  60 months         10.49             455    B3    Current  
2     29908.0  60 months          9.11             622    B2    Current  
3     13053.0  48 months         11.89             343    B3    Current  
4     24613.0  60 months         15.13             587    A3    Current  
Cleansed strings: trimmed and standardized text case


In [ ]:
# Fill missing values with mean for numeric columns only
loan_val = loan_clean.copy()

# Apply mean imputation
loan_val = loan_val.fillna(loan_val.mean(numeric_only=True))

# Verify missing values again
print(loan_val.isna().sum().sort_values(ascending=False).head(20))
print("Filled missing numeric values with column means")


loanId                0
memberId              0
date                  0
purpose               0
isJointApplication    0
loanAmount            0
term                  0
interestRate          0
monthlyPayment        0
grade                 0
loanStatus            0
dtype: int64
Filled missing numeric values with column means


In [ ]:
# Parse dates, extract features, convert flags and numerics
loan_tr = loan_val.copy()

if 'date' in loan_tr.columns:
    loan_tr['date'] = pd.to_datetime(loan_tr['date'], errors='coerce')
    loan_tr['year'] = loan_tr['date'].dt.year
    loan_tr['month'] = loan_tr['date'].dt.month

# Ensure numerics
for c in ['loanAmount','interestRate','monthlyPayment','term_months']:
    if c in loan_tr.columns:
        loan_tr[c] = pd.to_numeric(loan_tr[c], errors='coerce')

# Binary as int
if 'isJointApplication' in loan_tr.columns:
    loan_tr['isJointApplication'] = pd.to_numeric(loan_tr['isJointApplication'], errors='coerce').fillna(0).astype(int)

print(loan_tr.head())
print('Transformed dates and numeric fields')

    loanId  memberId       date            purpose  isJointApplication  \
0  1888978   2305095 2014-12-10  debtconsolidation                   0   
1  1299695   2610493 2014-09-15  debtconsolidation                   0   
2  1875016   2491679 2014-09-11  debtconsolidation                   0   
3  1440478   2092798 2016-04-22    homeimprovement                   0   
4  1124634   2633077 2016-02-03  debtconsolidation                   0   

   loanAmount       term  interestRate  monthlyPayment grade loanStatus  year  \
0     25190.0  60 months          6.25             490    E3    Current  2014   
1     21189.0  60 months         10.49             455    B3    Current  2014   
2     29908.0  60 months          9.11             622    B2    Current  2014   
3     13053.0  48 months         11.89             343    B3    Current  2016   
4     24613.0  60 months         15.13             587    A3    Current  2016   

   month  
0     12  
1      9  
2      9  
3      4  
4      2  
Tr

In [ ]:
# One-hot encode categorical features
from sklearn.preprocessing import OneHotEncoder

loan_enc_base = loan_tr.copy()
cat_cols = [c for c in ['purpose','grade','loanStatus'] if c in loan_enc_base.columns]
if len(cat_cols) > 0:
    enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded = enc.fit_transform(loan_enc_base[cat_cols])
    enc_cols = enc.get_feature_names_out(cat_cols)
    enc_df = pd.DataFrame(encoded, columns=enc_cols, index=loan_enc_base.index)
    loan_enc = pd.concat([loan_enc_base.drop(columns=cat_cols), enc_df], axis=1)
else:
    loan_enc = loan_enc_base

print(loan_enc.head())
print('Encoded categorical columns with one-hot')

    loanId  memberId       date  isJointApplication  loanAmount       term  \
0  1888978   2305095 2014-12-10                   0     25190.0  60 months   
1  1299695   2610493 2014-09-15                   0     21189.0  60 months   
2  1875016   2491679 2014-09-11                   0     29908.0  60 months   
3  1440478   2092798 2016-04-22                   0     13053.0  48 months   
4  1124634   2633077 2016-02-03                   0     24613.0  60 months   

   interestRate  monthlyPayment  year  month  ...  grade_C2  grade_C3  \
0          6.25             490  2014     12  ...       0.0       0.0   
1         10.49             455  2014      9  ...       0.0       0.0   
2          9.11             622  2014      9  ...       0.0       0.0   
3         11.89             343  2016      4  ...       0.0       0.0   
4         15.13             587  2016      2  ...       0.0       0.0   

   grade_D1  grade_D2  grade_D3  grade_E1  grade_E2  grade_E3  \
0       0.0       0.0      

In [ ]:
# Standardize selected numeric columns (z-score)
from sklearn.preprocessing import StandardScaler

loan_norm = loan_enc.copy()
num_cols = [c for c in ['loanAmount','interestRate','monthlyPayment','term_months','year','month'] if c in loan_norm.columns]

if len(num_cols) > 0:
    scaler = StandardScaler()
    loan_norm[num_cols] = scaler.fit_transform(loan_norm[num_cols])

print(loan_norm[num_cols].head() if len(num_cols) > 0 else loan_norm.head())
print('Normalized numeric columns (z-score) where applicable')

   loanAmount  interestRate  monthlyPayment      year     month
0    0.962571     -1.210878       -0.378313 -1.287148  1.653845
1    0.112683     -0.124934       -0.590343 -1.287148  0.757598
2    1.964764     -0.478378        0.421342 -1.287148  0.757598
3   -1.615558      0.233632       -1.268838  1.462933 -0.736148
4    0.840006      1.063457        0.209312  1.462933 -1.333646
Normalized numeric columns (z-score) where applicable


In [ ]:
# Drop high-cardinality IDs for modeling (keep a version with them if needed)
drop_cols = [c for c in ['loanId','memberId'] if c in loan_norm.columns]
loan_reduced = loan_norm.drop(columns=drop_cols) if len(drop_cols) > 0 else loan_norm.copy()

print(loan_reduced.head())
print('Reduced dataset by dropping identifiers not needed for modeling')

        date  isJointApplication  loanAmount       term  interestRate  \
0 2014-12-10                   0    0.962571  60 months     -1.210878   
1 2014-09-15                   0    0.112683  60 months     -0.124934   
2 2014-09-11                   0    1.964764  60 months     -0.478378   
3 2016-04-22                   0   -1.615558  48 months      0.233632   
4 2016-02-03                   0    0.840006  60 months      1.063457   

   monthlyPayment      year     month  purpose_auto  purpose_business  ...  \
0       -0.378313 -1.287148  1.653845           0.0               0.0  ...   
1       -0.590343 -1.287148  0.757598           0.0               0.0  ...   
2        0.421342 -1.287148  0.757598           0.0               0.0  ...   
3       -1.268838  1.462933 -0.736148           0.0               0.0  ...   
4        0.209312  1.462933 -1.333646           0.0               0.0  ...   

   grade_C2  grade_C3  grade_D1  grade_D2  grade_D3  grade_E1  grade_E2  \
0       0.0      

In [ ]:
# Placeholder: integrate other internal sources if available
# Example structure: loan_integrated = loan_reduced.merge(other_df, on='key', how='left')
loan_integrated = loan_reduced.copy()

print(loan_integrated.head())
print('Integration placeholder completed (no external/internal joins provided)')

        date  isJointApplication  loanAmount       term  interestRate  \
0 2014-12-10                   0    0.962571  60 months     -1.210878   
1 2014-09-15                   0    0.112683  60 months     -0.124934   
2 2014-09-11                   0    1.964764  60 months     -0.478378   
3 2016-04-22                   0   -1.615558  48 months      0.233632   
4 2016-02-03                   0    0.840006  60 months      1.063457   

   monthlyPayment      year     month  purpose_auto  purpose_business  ...  \
0       -0.378313 -1.287148  1.653845           0.0               0.0  ...   
1       -0.590343 -1.287148  0.757598           0.0               0.0  ...   
2        0.421342 -1.287148  0.757598           0.0               0.0  ...   
3       -1.268838  1.462933 -0.736148           0.0               0.0  ...   
4        0.209312  1.462933 -1.333646           0.0               0.0  ...   

   grade_C2  grade_C3  grade_D1  grade_D2  grade_D3  grade_E1  grade_E2  \
0       0.0      

In [ ]:
# Placeholder: derive additional features for enrichment
loan_enriched = loan_integrated.copy()

# Example derived features:
if set(['loanAmount','monthlyPayment','term_months']).issubset(loan_enriched.columns):
    # Approx total payments and naive interest paid proxy
    loan_enriched['approx_total_payment'] = loan_enriched['monthlyPayment'] * loan_enriched['term_months']
    loan_enriched['approx_interest_paid'] = loan_enriched['approx_total_payment'] - loan_enriched['loanAmount']

print(loan_enriched.head())
print('Enrichment: added simple derived features (if inputs available)')

        date  isJointApplication  loanAmount       term  interestRate  \
0 2014-12-10                   0    0.962571  60 months     -1.210878   
1 2014-09-15                   0    0.112683  60 months     -0.124934   
2 2014-09-11                   0    1.964764  60 months     -0.478378   
3 2016-04-22                   0   -1.615558  48 months      0.233632   
4 2016-02-03                   0    0.840006  60 months      1.063457   

   monthlyPayment      year     month  purpose_auto  purpose_business  ...  \
0       -0.378313 -1.287148  1.653845           0.0               0.0  ...   
1       -0.590343 -1.287148  0.757598           0.0               0.0  ...   
2        0.421342 -1.287148  0.757598           0.0               0.0  ...   
3       -1.268838  1.462933 -0.736148           0.0               0.0  ...   
4        0.209312  1.462933 -1.333646           0.0               0.0  ...   

   grade_C2  grade_C3  grade_D1  grade_D2  grade_D3  grade_E1  grade_E2  \
0       0.0      